# se-path-multi-v2

Historical `se-path-multi-v1` is not modified. This run optimises clipped synthetic mastery change, not the v1 quiz correctness bonus.
Training revision `se-path-multi-v2-coverage-r2`: every seed alternates Y3S1 and Y3S2. An even seed is not Y3S1-only, and an odd seed is not Y3S2-only.
Primary reported outcome is `mean_mastery_change`. Return is not accuracy.
Keep `RUN_FULL_TRAINING = False` until the intentional Colab run.
The notebook installs dependencies, uploads `se_path_multi_v2_colab.zip`, and finds the source root. It does not expect `/content/c4`.
Smoke writes a new timestamped folder. The full run also creates a new timestamped folder and does not overwrite an older `run_full` directory.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%pip install -q numpy gymnasium==1.4.0 stable-baselines3==2.9.0 sb3-contrib==2.9.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.3/476.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 6.7 MB/s eta 0:00:00


In [6]:
RUN_FULL_TRAINING = True
print('RUN_FULL_TRAINING', RUN_FULL_TRAINING)


RUN_FULL_TRAINING True


In [4]:
import shutil
import sys
import zipfile
from pathlib import Path

MARKER = Path('training_se_v2') / 'multi_action_v2' / 'smoke_runner.py'

def find_zip() -> Path:
    found = sorted(Path('/content').glob('se_path_multi_v2_colab.zip'))
    if found:
        return found[0]
    from google.colab import files
    uploaded = files.upload()
    zips = [Path('/content') / name for name in uploaded if name.endswith('.zip')]
    if not zips:
        raise FileNotFoundError('Upload se_path_multi_v2_colab.zip')
    return zips[0]

def find_source_root(start: Path) -> Path:
    if (start / MARKER).is_file():
        return start
    for path in start.rglob('smoke_runner.py'):
        if path.parent.name == 'multi_action_v2' and path.parent.parent.name == 'training_se_v2':
            return path.parents[2]
    raise FileNotFoundError(f'{MARKER} was not found in the extracted ZIP')

zip_path = find_zip()
extract_to = Path('/content/se_path_multi_v2_src')
if extract_to.exists():
    shutil.rmtree(extract_to)
extract_to.mkdir(parents=True)
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(extract_to)
ROOT = find_source_root(extract_to)
protocol_file = ROOT / 'training_se_v1' / 'protocol.py'
objective_file = ROOT / 'training_se_v2' / 'multi_action_v2' / 'objective.py'
if not protocol_file.is_file() or not objective_file.is_file():
    raise FileNotFoundError('extracted ZIP is missing protocol.py or objective.py')
sys.path[:] = [str(ROOT)] + [entry for entry in sys.path if entry not in ('', str(ROOT))]
print('ZIP', zip_path)
print('ROOT', ROOT)


Saving se_path_multi_v2_colab.zip to se_path_multi_v2_colab.zip
ZIP /content/se_path_multi_v2_colab.zip
ROOT /content/se_path_multi_v2_src


In [7]:
import training_se_v1.protocol as se_protocol
from training_se_v2.multi_action_v2.smoke_runner import main
print('protocol module', se_protocol.__file__)
if not str(se_protocol.__file__).startswith(str(ROOT)):
    raise RuntimeError('protocol resolved outside the extracted ZIP')
main()


protocol module /content/se_path_multi_v2_src/training_se_v1/protocol.py
coverage seed 0 Y3S1_episodes 2 Y3S1_steps 80 Y3S2_episodes 2 Y3S2_steps 48
coverage seed 1 Y3S1_episodes 2 Y3S1_steps 80 Y3S2_episodes 2 Y3S2_steps 48
coverage seed 2 Y3S1_episodes 2 Y3S1_steps 80 Y3S2_episodes 2 Y3S2_steps 48
OK se-path-multi-v2 obs 101 actions 184 smoke_not_activated se-path-multi-v2-coverage-r2


In [8]:
if 'ROOT' not in dir():
    raise RuntimeError('Run the ZIP extraction cell first.')
if not RUN_FULL_TRAINING:
    print('FULL TRAINING SKIPPED. Smoke does not approve a policy.')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    from training_se_v2.multi_action_v2.full_run import run_full
    parent = Path('/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_multi_v2')
    manifest = run_full(parent, timesteps=80_000, episodes=20)
    print('training_revision', manifest['training_revision'])
    print('semester_coverage', manifest['semester_coverage'])
    print('criterion_met', manifest['criterion_met'])
    print('activation_allowed', manifest['activation_allowed'])
    print('serves_students', manifest['serves_students'])
    print('run_dir', manifest['run_dir'])
    print('export', Path(manifest['run_dir']) / 'export')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
criterion_met True activation_allowed True serves_students False run_dir /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_multi_v2/se-path-multi-v2-coverage-r2_20261017T040345500695Z
training_revision se-path-multi-v2-coverage-r2
semester_coverage {'0': {'Y3S1': {'episodes': 1002, 'steps': 40080}, 'Y3S2': {'episodes': 1002, 'steps': 40048}}, '1': {'Y3S1': {'episodes': 1002, 'steps': 40080}, 'Y3S2': {'episodes': 1002, 'steps': 40048}}, '2': {'Y3S1': {'episodes': 1002, 'steps': 40080}, 'Y3S2': {'episodes': 1002, 'steps': 40048}}}
criterion_met True
activation_allowed True
serves_students False
run_dir /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_multi_v2/se-path-multi-v2-coverage-r2_20261017T040345500695Z
export /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_multi_v2/se-path-multi-v2-coverage-r2_20261017T040345500695Z/export


In [9]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

RUN = Path(
    "/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/"
    "se_path_multi_v2/"
    "se-path-multi-v2-coverage-r2_20261017T040345500695Z"
)
EXPORT = RUN / "export"

selection_path = EXPORT / "selection_val.json"
assert selection_path.exists(), f"Missing: {selection_path}"

selection = json.loads(selection_path.read_text())
selected_seed = str(selection["selected_seed"])
print("Selected seed:", selected_seed)

rows = []
for semester, methods in selection["by_seed"][selected_seed].items():
    for method, metrics in methods.items():
        def mean(key):
            value = metrics.get(key)
            return value.get("mean") if isinstance(value, dict) else value

        rows.append({
            "semester": semester,
            "method": method,
            "mastery_change": mean("mean_mastery_change"),
            "weakest_topic_progress": mean("weakest_topic_progress"),
            "return": mean("return"),
            "invalid_action_rate": mean("invalid_action_rate"),
            "no_progress_rate": mean("no_progress_rate"),
        })

display(pd.DataFrame(rows))
print("Evidence: synthetic simulator validation; not accuracy.")

Selected seed: 1


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,semester,method,mastery_change,weakest_topic_progress,return,invalid_action_rate,no_progress_rate
0,Y3S1,maskable_ppo,0.082957,0.170,1.923108,0.0,0.05875
1,Y3S1,weakest_practice,0.074088,0.171,1.700000,0.0,0.00000
2,Y3S1,recent_performance,0.085371,0.073,1.967000,0.0,0.00000
3,Y3S1,weakest_multi,0.072609,0.165,1.665000,0.0,0.00000
4,Y3S1,random_valid,0.077059,0.077,1.773000,0.0,0.00000
5,Y3S2,maskable_ppo,0.130406,0.142,2.046294,0.0,0.02000
6,Y3S2,weakest_practice,0.114199,0.220,1.790000,0.0,0.00000
7,Y3S2,recent_performance,0.127731,0.215,2.006000,0.0,0.00000
8,Y3S2,weakest_multi,0.110539,0.203,1.735000,0.0,0.00000
9,Y3S2,random_valid,0.114945,0.121,1.804000,0.0,0.00000


Evidence: synthetic simulator validation; not accuracy.


In [11]:
from pathlib import Path
from datetime import datetime, timezone
from google.colab import files
import hashlib
import json
import zipfile

RUN = Path(
    "/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/"
    "se_path_multi_v2/"
    "se-path-multi-v2-coverage-r2_20261017T040345500695Z"
)
assert (RUN / "export" / "manifest.json").is_file(), RUN

# Find the source loaded in this session.
import training_se_v2.multi_action_v2.full_run as full_run_module

SOURCE = Path(full_run_module.__file__).resolve().parents[2]
assert (SOURCE / "training_se_v2").is_dir(), SOURCE

print("Upload the executed 03_se_path_multi_v2.ipynb downloaded from Colab.")
uploaded = files.upload()
notebooks = {
    name: data for name, data in uploaded.items()
    if name.lower().endswith(".ipynb")
}
assert len(notebooks) == 1, "Upload exactly one executed .ipynb file."

nb_name, nb_bytes = next(iter(notebooks.items()))
nb = json.loads(nb_bytes.decode("utf-8"))
code_cells = [c for c in nb["cells"] if c.get("cell_type") == "code"]
output_cells = sum(bool(c.get("outputs")) for c in code_cells)
assert output_cells > 0, "Notebook has no saved outputs; download the executed notebook."
print("Notebook code cells:", len(code_cells))
print("Cells with saved outputs:", output_cells)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
DEST = Path("/content/drive/MyDrive/AdaptiveLearnSE_Backups")
DEST.mkdir(parents=True, exist_ok=True)
ZIP = DEST / f"C4_multi_v2_coverage_r2_complete_{stamp}.zip"

excluded_dirs = {
    "__pycache__", ".git", ".venv", "venv", "node_modules",
    ".ipynb_checkpoints"
}
entries = {}

def add_tree(root, prefix):
    if not root.exists():
        return
    paths = [root] if root.is_file() else sorted(root.rglob("*"))
    for p in paths:
        if not p.is_file() or p.is_symlink():
            continue
        if any(part in excluded_dirs for part in p.parts):
            continue
        if p.name.startswith(".env") or p.suffix == ".pyc":
            continue
        rel = p.name if root.is_file() else p.relative_to(root).as_posix()
        entries[f"{prefix}/{rel}"] = p

# All models, checkpoints, metrics, configs and saved data in this run.
add_tree(RUN, f"run/{RUN.name}")

# Source needed to reproduce synthetic data and training.
for name in ("training_se_v2", "training_se_v1", "catalogues"):
    add_tree(SOURCE / name, f"source/{name}")

for name in (
    "README_SE_MULTI_V2_COLAB.md",
    "requirements.txt",
    "requirements-training.txt",
):
    add_tree(SOURCE / name, "source")

# Include any dataset files shipped in this source package.
add_tree(SOURCE / "datasets", "source/datasets")

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

inventory = []
nb_archive = "notebooks/03_se_path_multi_v2_executed.ipynb"

with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for archive_name, path in sorted(entries.items()):
        z.write(path, archive_name)
        inventory.append({
            "path": archive_name,
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })

    z.writestr(nb_archive, nb_bytes)
    inventory.append({
        "path": nb_archive,
        "bytes": len(nb_bytes),
        "sha256": hashlib.sha256(nb_bytes).hexdigest(),
    })

    z.writestr("BACKUP_MANIFEST.json", json.dumps({
        "run_dir": str(RUN),
        "source_dir": str(SOURCE),
        "training_revision": "se-path-multi-v2-coverage-r2",
        "synthetic": True,
        "notebook_cells_with_outputs": output_cells,
        "files": inventory,
        "data_note": (
            "Includes saved data if present. Simulator observations are "
            "generated during rollouts; generator source/config/seeds are "
            "included. No new dataset or training was created by packaging."
        ),
    }, indent=2))

with zipfile.ZipFile(ZIP) as z:
    assert z.testzip() is None, "ZIP integrity check failed."
    for row in inventory:
        data = z.read(row["path"])
        assert hashlib.sha256(data).hexdigest() == row["sha256"], row["path"]

print("Verified files:", len(inventory))
print("Package:", ZIP)
print("Size MB:", round(ZIP.stat().st_size / 1024**2, 2))
files.download(str(ZIP))

Upload the executed 03_se_path_multi_v2.ipynb downloaded from Colab.


AssertionError: Upload exactly one executed .ipynb file.

## Selection and export

Seeds 0, 1 and 2. Each seed trains on both semesters. Select the highest validation `mean_mastery_change` across Y3S1 and Y3S2 on the shared bank (`81000` / `82000`). Do not use the held-out bank (`91000` / `92000`) while choosing.
A full export without both semesters for every seed is rejected. `criterion_met` is still the mastery and invalid-action floor. It is not lowered.
If `criterion_met` is false, do not import the export.
If it is true, copy that run's `export/` to `ai-services/c4/runtime-artifacts/se_path_multi_v2/active/` only. The student plan does not call this endpoint.
